# Complex Systems QLoRA: evaluation (Kaggle, one T4)

Generates test-split predictions for **(a) the base model** and **(b) base + LoRA** with
transformers + PEFT (greedy, batch 1), then scores them into `results/<run>.json`
(aggregate metrics and the test split's SHA-256 only, no text).

**Before Run All**
1. *Add Input*: the private dataset holding `reviewed.jsonl`.
2. The adapter: run this in the training session, or *Add Input* the training notebook's output
   (it looks for `adapter_config.json`), or leave it to download from the Hub repo in
   `config/eval.yaml`. Set `ADAPTER` below to force a path or repo id.
3. *Accelerator*: **GPU T4 x2** (one is used). *Internet*: **on**.

Keep this notebook **private**: `/kaggle/working/predictions` holds model text about private passages.

In [ ]:
import os
import subprocess
import sys
from importlib import metadata

# Use one T4 of Kaggle's pair. Set before anything imports torch.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["APP_ENV"] = "local"

REPO_URL = "https://github.com/WaiteM124/complex-systems-lora.git"
REPO_REF = "main"  # or a commit SHA / tag for a reproducible run
REPO_DIR = "/tmp/complex-systems-lora"


def run(*args: str) -> None:
    """Run a command in the repo and stream its output; raise if it fails."""
    proc = subprocess.Popen(
        list(args), cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"command failed: {' '.join(args)}")


RUN_NAME = "m3-qlora-r16"  # results/<RUN_NAME>.json
ADAPTER = ""  # empty = auto: /kaggle/working/adapter, then /kaggle/input, then the Hub

In [ ]:
subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"], check=False)

In [ ]:
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--quiet", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "--quiet", REPO_REF], check=True)
commit = subprocess.run(
    ["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True, check=True
).stdout.strip()
print("repo commit:", commit)

In [ ]:
# Pinned GPU stack from the repo's `train` extra (pyproject.toml). First try to keep
# the Kaggle image's own torch build (the constraint stops pip replacing it); if that
# torch is outside unsloth's supported range, let pip install a compatible torch.
torch_version = metadata.version("torch").split("+")[0]
with open("/tmp/constraints.txt", "w", encoding="utf-8") as handle:
    handle.write(f"torch=={torch_version}\n")
pip = [sys.executable, "-m", "pip", "install", "--quiet"]
try:
    subprocess.run([*pip, "-c", "/tmp/constraints.txt", "-e", f"{REPO_DIR}[train]"], check=True)
    print("kept Kaggle torch", torch_version)
except subprocess.CalledProcessError:
    print(f"torch {torch_version} does not fit the pins; installing a compatible torch")
    subprocess.run([*pip, "-e", f"{REPO_DIR}[train]"], check=True)
for name in (
    "torch",
    "unsloth",
    "transformers",
    "trl",
    "peft",
    "bitsandbytes",
    "accelerate",
    "datasets",
):
    print(f"{name}=={metadata.version(name)}")

In [ ]:
# (a) base model
run(sys.executable, "-m", "app.eval.local", "--system", "base")

In [ ]:
# (b) base + LoRA adapter
extra = ["--adapter", ADAPTER] if ADAPTER else []
run(sys.executable, "-m", "app.eval.local", "--system", "lora", *extra)

In [ ]:
run(sys.executable, "-m", "app.eval.score", "--kaggle", "--run", RUN_NAME)

## Next (on your PC)

1. Download `/kaggle/working/results/<RUN_NAME>.json` and commit it to `results/` in a PR
   (aggregates only). `make eval` then gates LoRA against base.
2. Download `/kaggle/working/predictions/*.jsonl` into `data/private/predictions/` (gitignored,
   **never commit**) to run the Bedrock baseline and judge, then re-score the same run:
   `python -m app.eval.score --run <RUN_NAME>`.